# DVB-S2 ACM — DQN Training on GPU
Train the Dueling Double DQN agent on Colab GPU, then download `dqn_acm_model.pt` back to your machine.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# Cell 1: Clone repo and install deps
!git clone https://github.com/Manujayaugp/gr-dvbs2acm.git /content/gr-dvbs2acm
!pip install -q torch numpy scipy matplotlib

In [ ]:
# Cell 2: Setup paths
import sys, os
os.chdir('/content/gr-dvbs2acm')
sys.path.insert(0, 'python')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 3: Import modules (no GNU Radio needed)
from dvbs2acm.acm_controller_ai import DQNAgent, ChannelFeatures, Transition, TORCH_AVAILABLE
from dvbs2acm.leo_channel_model import LeoChannelModel, LeoOrbitParams
print(f'TORCH_AVAILABLE: {TORCH_AVAILABLE}')

In [ ]:
# Cell 4: Simulation infrastructure (from acm_loopback_sim.py)
import numpy as np
from collections import deque
from typing import List, Tuple, Optional

SYMBOL_RATE_HZ   = 5e6
PL_FRAME_SYMBOLS = 33282
FRAME_DT_S       = PL_FRAME_SYMBOLS / SYMBOL_RATE_HZ
HYSTERESIS_DB    = 1.5
DOWNGRADE_MARGIN_DB = 1.0
SNR_EST_STD_DB   = 0.3
SNR_AVG_WINDOW   = 15

_MODCODS = [
    (1,  'QPSK 1/4',    -2.35, 0.490),
    (2,  'QPSK 1/3',    -1.24, 0.657),
    (3,  'QPSK 2/5',    -0.30, 0.789),
    (4,  'QPSK 1/2',     1.00, 0.988),
    (5,  'QPSK 3/5',     2.23, 1.188),
    (6,  'QPSK 2/3',     3.10, 1.322),
    (7,  'QPSK 3/4',     4.03, 1.487),
    (8,  'QPSK 4/5',     4.68, 1.587),
    (9,  'QPSK 5/6',     5.18, 1.655),
    (10, 'QPSK 8/9',     6.20, 1.766),
    (11, 'QPSK 9/10',    6.42, 1.789),
    (12, '8PSK 3/5',     5.50, 1.980),
    (13, '8PSK 2/3',     6.62, 2.228),
    (14, '8PSK 3/4',     7.91, 2.479),
    (15, '8PSK 5/6',     9.35, 2.642),
    (16, '8PSK 8/9',    10.69, 2.646),
    (17, '8PSK 9/10',   10.98, 2.679),
    (18, '16APSK 2/3',  11.03, 2.637),
    (19, '16APSK 3/4',  12.73, 2.966),
    (20, '16APSK 4/5',  13.64, 3.165),
    (21, '16APSK 5/6',  14.28, 3.300),
    (22, '16APSK 8/9',  15.69, 3.523),
    (23, '16APSK 9/10', 16.05, 3.567),
    (24, '32APSK 3/4',  16.05, 3.703),
    (25, '32APSK 4/5',  17.04, 3.952),
    (26, '32APSK 5/6',  17.73, 4.120),
    (27, '32APSK 8/9',  19.57, 4.397),
    (28, '32APSK 9/10', 20.14, 4.453),
]

def ber_from_margin(margin_db):
    if margin_db >= 1.5:  return 1e-9
    if margin_db >= 0.0:  return 1e-7
    if margin_db >= -0.5: return 1e-5
    if margin_db >= -1.0: return 1e-3
    if margin_db >= -1.5: return 1e-2
    return 0.5

def fer_from_ber(ber, kbch=32208):
    if ber < 1e-12: return 0.0
    return min(1.0, 1.0 - (1.0 - ber) ** kbch)

def _modcod_for_snr_rule(snr_db, current_id):
    best = 1
    for mc in _MODCODS:
        mid, thr = mc[0], mc[2]
        if mid <= current_id:
            if snr_db >= thr + DOWNGRADE_MARGIN_DB: best = mid
        else:
            if snr_db >= thr + HYSTERESIS_DB: best = mid
    return best

print('Simulation infrastructure ready')

In [ ]:
# Cell 5: Scenario generators
def scenario_sweep(snr_start=20.0, snr_end=-3.0, steps=200):
    snr = (np.linspace(snr_start, snr_end, steps // 2).tolist()
           + np.linspace(snr_end, snr_start, steps // 2).tolist())
    ch = [ChannelFeatures() for _ in snr]
    return snr, ch, 1.0

def scenario_rain_fade(initial_snr=18.0, fade_db=10.0, duration_s=60.0, seed=42):
    rng = np.random.default_rng(seed)
    dt = FRAME_DT_S * 10
    n = int(duration_s / dt)
    t = np.linspace(0, duration_s, n)
    fade = np.zeros(n)
    t1, t2 = 0.25 * duration_s, 0.75 * duration_s
    for i, ti in enumerate(t):
        if ti < t1: fade[i] = 0.0
        elif ti < t1 + 5: fade[i] = fade_db * (ti - t1) / 5.0
        elif ti < t2 - 5: fade[i] = fade_db
        elif ti < t2: fade[i] = fade_db * (t2 - ti) / 5.0
        else: fade[i] = 0.0
    snr = (initial_snr - fade + rng.normal(0, SNR_EST_STD_DB, n)).tolist()
    ch = [ChannelFeatures(elevation_deg=60.0, pass_fraction=float(i)/max(n-1,1),
          rain_db=float(fade[i]), rtt_ms=6.0) for i in range(n)]
    return snr, ch, dt

def scenario_leo(altitude_km=500.0, rain_rate_mm_hr=5.0, seed=None):
    params = LeoOrbitParams(altitude_km=altitude_km, freq_hz=8.025e9,
        tx_power_dbw=0.0, tx_gain_dbi=10.0, rx_gain_dbi=37.0,
        system_noise_temp_k=150.0, rain_rate_mm_hr=rain_rate_mm_hr, symbol_rate_hz=5e6)
    rng = np.random.default_rng(seed)
    model = LeoChannelModel(params)
    dt = FRAME_DT_S * 10
    states = model.simulate_pass(dt_s=dt, rng=rng)
    if not states: return [10.0]*100, [ChannelFeatures()]*100, dt
    total_t = max(states[-1].time_s, 1e-6)
    snr = [s.snr_db for s in states]
    ch = [ChannelFeatures(elevation_deg=s.elevation_deg,
          pass_fraction=float(s.time_s/total_t),
          doppler_rate_hz_s=s.doppler_rate_hz_s,
          rain_db=s.rain_atten_db, rtt_ms=s.rtt_ms) for s in states]
    return snr, ch, dt

print('Scenarios ready')

In [ ]:
# Cell 6: Simulation loop
def run_simulation(snr_profile, dt_s, strategy='rule', agent=None,
                   channel_features=None, train=True):
    use_dqn = (strategy == 'dqn' and agent is not None)
    current_id = 4
    snr_history = deque([snr_profile[0]] * 16, maxlen=16)
    snr_avg_buf = deque([snr_profile[0]] * SNR_AVG_WINDOW, maxlen=SNR_AVG_WINDOW)
    log_eta, log_ber, log_fer, log_mid = [], [], [], []
    prev_sv, prev_action, prev_id = None, None, current_id

    for step_idx, snr_db in enumerate(snr_profile):
        ch = (channel_features[step_idx] if channel_features and step_idx < len(channel_features)
              else ChannelFeatures())
        snr_meas = snr_db + np.random.normal(0, SNR_EST_STD_DB)
        snr_history.append(snr_meas)
        snr_avg_buf.append(snr_meas)
        snr_for_acm = float(np.mean(snr_avg_buf))

        if strategy == 'ccm': new_id = 4
        elif strategy == 'rule': new_id = _modcod_for_snr_rule(snr_for_acm, current_id)
        elif use_dqn:
            sv = agent.build_state(list(snr_history), current_id,
                log_ber[-1] if log_ber else 1e-9,
                log_fer[-1] if log_fer else 0.0, ch)
            new_id = agent.select_action(sv, snr_for_acm, current_id) + 1
        else: new_id = _modcod_for_snr_rule(snr_for_acm, current_id)
        new_id = max(1, min(28, new_id))

        mc = _MODCODS[new_id - 1]
        eta, thr = mc[3], mc[2]
        ber = ber_from_margin(snr_db - thr)
        fer = fer_from_ber(ber)

        if use_dqn and train and prev_sv is not None and prev_action is not None:
            reward = agent.compute_reward(prev_action - 1, snr_db, prev_id,
                log_fer[-1] if log_fer else 0.0, log_ber[-1] if log_ber else 1e-9)
            agent.push_experience(Transition(prev_sv, prev_action-1, reward, sv, False))
            agent.train_step()

        if use_dqn:
            prev_sv, prev_action, prev_id = sv, new_id, current_id

        current_id = new_id
        log_eta.append(eta); log_ber.append(ber)
        log_fer.append(fer); log_mid.append(new_id)

    avg_eta = np.mean(log_eta)
    qef_pct = np.mean(np.array(log_ber) < 1e-7) * 100
    switches = int(np.sum(np.diff(log_mid) != 0))
    return dict(avg_eta=avg_eta, qef_pct=qef_pct, switches=switches)

print('Simulation loop ready')

In [ ]:
# Cell 7: Train!
import time

EPISODES = 1000
TARGET_EPS = 0.01
MODEL_PATH = '/content/gr-dvbs2acm/dqn_acm_model.pt'

agent = DQNAgent(model_path=MODEL_PATH, epsilon_decay=0.99995)
print(f'Start: eps={agent.epsilon:.4f}, steps={agent.train_steps}')

scenarios = ['sweep', 'leo', 'rain_fade']
t0 = time.time()

for ep in range(1, EPISODES + 1):
    sc = scenarios[ep % 3]
    seed = ep
    rng = np.random.default_rng(seed)

    if sc == 'sweep':
        snr_profile, ch, dt = scenario_sweep(rng.uniform(15,22), rng.uniform(-5,0), 200)
    elif sc == 'leo':
        snr_profile, ch, dt = scenario_leo(500.0, rng.uniform(0,15), seed)
    else:
        snr_profile, ch, dt = scenario_rain_fade(rng.uniform(14,20), rng.uniform(5,15), 60.0, seed)

    res = run_simulation(snr_profile, dt, 'dqn', agent, ch, train=True)

    if ep % 25 == 0 or ep == 1:
        elapsed = time.time() - t0
        eta_left = (EPISODES - ep) * elapsed / max(ep, 1)
        print(f'[EP {ep:4d}/{EPISODES}] eps={agent.epsilon:.4f} '
              f'steps={agent.train_steps:7d} eta={res["avg_eta"]:.3f} '
              f'QEF={res["qef_pct"]:.1f}% sc={sc} '
              f'elapsed={elapsed:.0f}s ETA={eta_left:.0f}s')

    if ep % 100 == 0:
        agent.save_model()
        print(f'  [SAVED] eps={agent.epsilon:.4f} steps={agent.train_steps}')

    if agent.epsilon <= TARGET_EPS:
        print(f'\n*** CONVERGED at ep {ep}: eps={agent.epsilon:.5f} ***')
        agent.save_model()
        break

agent.save_model()
elapsed = time.time() - t0
print(f'\nDone: eps={agent.epsilon:.4f}, steps={agent.train_steps}, time={elapsed:.0f}s')

In [ ]:
# Cell 8: Evaluate converged model
agent_eval = DQNAgent(model_path=MODEL_PATH)
agent_eval.epsilon = 0.0  # pure greedy for evaluation
print(f'Eval agent: steps={agent_eval.train_steps}, eps={agent_eval.epsilon}')

for sc_name, sc_fn in [('sweep', lambda: scenario_sweep()),
                        ('leo', lambda: scenario_leo(500.0, 5.0, seed=999)),
                        ('rain_fade', lambda: scenario_rain_fade())]:
    snr, ch, dt = sc_fn()
    r_ccm  = run_simulation(snr, dt, 'ccm',  train=False)
    r_rule = run_simulation(snr, dt, 'rule', train=False)
    r_dqn  = run_simulation(snr, dt, 'dqn', agent_eval, ch, train=False)
    print(f'\n--- {sc_name.upper()} ---')
    print(f'{"Strategy":<20} {"eta":>8} {"QEF%":>8} {"Switches":>8}')
    for label, r in [('CCM', r_ccm), ('Rule-based', r_rule), ('DQN', r_dqn)]:
        print(f'{label:<20} {r["avg_eta"]:8.3f} {r["qef_pct"]:7.1f}% {r["switches"]:8d}')

In [ ]:
# Cell 9: Download trained model
from google.colab import files
files.download('/content/gr-dvbs2acm/dqn_acm_model.pt')
print('Download dqn_acm_model.pt and copy to gr-dvbs2acm/ on your machine')